# Full-dataset embeddings

This notebook resolves external identifiers for the Metabolite, Protein and GeneProduct nodes of the full graph (`data/processed/`), computes their embeddings, attaches them to `heterodata.pt` and builds the embedding-aware taxa split.

| Node type | Source identifiers | Embedding |
|---|---|---|
| `Metabolite` | InChIKey / ChEBI / MetaNetX / PubChem / KEGG / HMDB | SMILES → MAP4 fingerprint |
| `Protein` | UniProt / TAIR / PlantCyc / PDB / RefSeq | sequence → ESM-C (`esmc`) |
| `GeneProduct` | GenBank / TAIR / PlantCyc / UniProt / STRING / Phytozome / BioCyc / Ensembl | DNA sequence → PlantCaduceus |

Three properties of the data shape the workflow:

1. **Species coverage.** The graph spans hundreds of organisms, most of them sparsely annotated in public databases, so the resolution rate varies by species and by namespace. The diagnostics in §4 and §5 measure it on a sample before the full runs.
2. **Locus tags.** `extract_locus` (`LOCUS_RE`) only recognises Arabidopsis (`AT#G#####`) and soybean (`Glyma.##G######`) locus-tag formats. For `tair.name` and `plantcyc` nodes whose identifier contains neither, the raw identifier is searched in UniProt as a gene name, which normally finds nothing. §4.3 quantifies the effect on Protein resolution.
3. **Run log.** Every resolve and embed run appends a record (`node_type`, `stage`, `total`, `resolved` or `embedded`, `timestamp_utc`) to `data/interim/full_embeddings/embedding_log.jsonl`. §6 reads this file, so the coverage numbers do not depend on which terminal or kernel produced them.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

FULL_DIR = ROOT / "data" / "processed"
CACHE_DIR = ROOT / "data" / "interim" / "full_embeddings"
INTERIM_DIR = ROOT / "data" / "interim"
LOG_PATH = CACHE_DIR / "embedding_log.jsonl"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "scripts"))
import embed_common as common
import embed_metabolites
import embed_proteins
import embed_genes

_required = [FULL_DIR / "nodes.tsv", FULL_DIR / "edges.tsv", FULL_DIR / "heterodata.pt"]
_missing = [p for p in _required if not p.exists()]
for p in _required:
    print(f"  {p}: {'OK' if p.exists() else 'MISSING'}")
if _missing:
    print()
    print("Rebuild the full graph first (see the markdown cell below for why), then re-run this cell:")
    print()
    print("  python scripts/fetch_zenodo.py")
    print("  python scripts/unpack_ttl.py")
    print("  python scripts/inspect_rdf.py --top 25")
    print("  python scripts/rdf_to_typed_tables.py")
    print("  python scripts/build_pyg.py")


## 1. Node-id namespace inventory (full graph)

In [ ]:
nodes = pd.read_csv(FULL_DIR / "nodes.tsv", sep="\t")

for node_type in ["Metabolite", "Protein", "GeneProduct"]:
    ids = nodes.loc[nodes["node_type"] == node_type, "node_id"]
    counts = Counter(common.namespace(n) for n in ids)
    print(f"{node_type} ({len(ids)} nodes):")
    for ns, n in counts.most_common():
        print(f"    {ns:<20} {n}")
    print()


## 2. Organism coverage

Every Metabolite, Protein and GeneProduct node has a direct `organism` edge, which `load_organism_map` resolves. For graphs without direct tags it falls back to `Protein/GeneProduct -is_part_of-> Pathway -organism-> Organism`; this path is not needed here.

NCBI taxon `33090` (Viridiplantae, i.e. green plants in general) is the most common `organism` edge target, because the upstream taxonomy annotations do not assign those nodes to a more specific species. There is no species to resolve a sequence against, so these nodes are expected to stay unresolved.

In [ ]:
organism_map = common.load_organism_map(FULL_DIR)

for node_type in ["Protein", "GeneProduct"]:
    ids = nodes.loc[nodes["node_type"] == node_type, "node_id"]
    with_organism = sum(n in organism_map for n in ids)
    print(f"{node_type}: {with_organism}/{len(ids)} have an organism edge")

taxon_counts = Counter(organism_map.values())
print(f"\n{len(taxon_counts)} distinct taxa across Protein+GeneProduct organism edges")
print("Top 10 by node count:")
for taxon, n in taxon_counts.most_common(10):
    flag = "  <- generic 'Viridiplantae', not a real species" if taxon == "33090" else ""
    print(f"  taxid:{taxon:<10} {n:>5}{flag}")


## 3. Metabolite resolution

The graph-native steps are not rate-limited, so all Metabolite nodes are resolved directly in the notebook.

In [ ]:
# Parse reactions.ttl / properties_extra.ttl into a cache of graph-native
# properties (SMILES and cross-references) per identifiers.org URI.
xrefs = common.load_properties_extra_xrefs(INTERIM_DIR, CACHE_DIR)
print(f"properties_extra_xrefs.json: {len(xrefs):,} identifiers.org URIs with graph-native properties")

metabolite_ids = common.load_nodes(FULL_DIR, "Metabolite")
metabolite_by_namespace = {}
for node_id in metabolite_ids:
    metabolite_by_namespace.setdefault(common.namespace(node_id), []).append(node_id)

print(f"\nMetabolite namespace x property coverage ({len(metabolite_ids)} total):")
print(f"  {'namespace':<20} {'total':>6}  {'Smiles':>8}  {'AltPubchem':>10}  {'AltMnx':>8}  {'NoStruct':>8}")
for ns, ids in metabolite_by_namespace.items():
    has_smiles  = sum(bool(xrefs.get(n, {}).get("Smiles")) for n in ids)
    has_pubchem = sum(bool(xrefs.get(n, {}).get("AlternativeId_Pubchem")) for n in ids)
    has_mnx     = sum(bool(xrefs.get(n, {}).get("AlternativeId_Metanetx")) for n in ids)
    no_struct   = sum(xrefs.get(n, {}).get("HasNoStructure?") == "T" for n in ids)
    print(f"  {ns:<20} {len(ids):>6}  {has_smiles:>8}  {has_pubchem:>10}  {has_mnx:>8}  {no_struct:>8}")


In [ ]:
# Graph-native steps (1-3) need no network. The PubChem/KEGG REST fallbacks
# (steps 4-6) only run for nodes not covered by properties_extra, so the full
# resolve can still take a few minutes.
smiles = embed_metabolites.resolve_metabolite_smiles(metabolite_ids, CACHE_DIR, interim_dir=INTERIM_DIR)
print(f"resolved SMILES for {len(smiles)}/{len(metabolite_ids)} Metabolite nodes")
common.log_event(LOG_PATH, node_type="Metabolite", stage="resolve",
                  total=len(metabolite_ids), resolved=len(smiles))

print(f"\nPer-namespace resolution rate:")
for ns, ids in metabolite_by_namespace.items():
    n_ok = sum(n in smiles for n in ids)
    print(f"  {ns:<20} {n_ok:>4}/{len(ids):<4}  ({100*n_ok/len(ids):.0f}%)")


### Why are the unresolved metabolites unresolved?

`resolve_metabolite_smiles` writes the ids it could not resolve to `data/interim/full_embeddings/metabolite_unresolved.txt`. The cell below classifies them. Most should be abstract compound classes (`HasNoStructure?=T`) for which no structure exists, not resolver failures.

In [ ]:
unresolved_path = CACHE_DIR / "metabolite_unresolved.txt"
unresolved_ids = unresolved_path.read_text().splitlines() if unresolved_path.exists() else []

no_struct, maybe_fixable = [], []
for uid in unresolved_ids:
    props = xrefs.get(uid, {})
    ns    = common.namespace(uid)
    ident = common.local_id(uid)
    if props.get("HasNoStructure?") == "T":
        no_struct.append((ns, ident))
    else:
        maybe_fixable.append((ns, ident, props))

print(f"Unresolved: {len(unresolved_ids)}")
print(f"  HasNoStructure?=T (abstract compound class, no SMILES possible anywhere): {len(no_struct)}")
print(f"  no HasNoStructure? flag (edge case, worth a closer look): {len(maybe_fixable)}")
if maybe_fixable:
    print("\n  Edge cases:")
    for ns, ident, props in maybe_fixable[:20]:
        print(f"    {ns}/{ident}  xrefs={dict(props)}")


### MAP4 fingerprint

`MAP4Calculator(is_folded=True)` gives a 1,024-dim binary fingerprint (`int64`; cast to `float32` before feeding a GNN). It runs on CPU and is quick once the SMILES are resolved, so it is computed inline.

In [ ]:
try:
    metabolite_embeddings = embed_metabolites.compute_metabolite_embeddings(smiles)
    out_path = FULL_DIR / "embeddings_metabolite.pt"
    import torch
    torch.save(metabolite_embeddings, out_path)
    print(f"saved {len(metabolite_embeddings)} embeddings -> {out_path}")
    common.log_event(LOG_PATH, node_type="Metabolite", stage="embed",
                      total=len(metabolite_ids), embedded=len(metabolite_embeddings), out_path=str(out_path))
except ImportError as e:
    print(f"MAP4 deps not installed ({e}) -- run: conda env update -f environment.yml")


### 3.2 Alternative: run via CLI

The cells above resolve and embed the metabolites inline. To run them from the command line, as for the protein and gene steps below:

```bash
conda activate plantmetbench-data

python scripts/embed_metabolites.py \
    --data-dir data/processed --cache-dir data/interim/full_embeddings \
    --resolve-only

python scripts/embed_metabolites.py \
    --data-dir data/processed --cache-dir data/interim/full_embeddings
# -> data/processed/embeddings_metabolite.pt
```

## 4. Protein resolution diagnostic (sample)

Protein resolution makes live UniProt and RCSB calls, so it is diagnosed first on a stratified sample per namespace. The full run uses the command-line script (§4.2).

In [ ]:
SAMPLE_PER_NAMESPACE = 8

protein_ids = common.load_nodes(FULL_DIR, "Protein")
protein_by_namespace = {}
for node_id in protein_ids:
    protein_by_namespace.setdefault(common.namespace(node_id), []).append(node_id)

sample_protein_ids = []
for ns, ids in protein_by_namespace.items():
    sample_protein_ids.extend(ids[:SAMPLE_PER_NAMESPACE])
    print(f"  sampling {min(SAMPLE_PER_NAMESPACE, len(ids))}/{len(ids)} '{ns}' Protein ids")

protein_sample_sequences = embed_proteins.resolve_protein_sequences(sample_protein_ids, organism_map, CACHE_DIR)
print(f"\nresolved sequences for {len(protein_sample_sequences)}/{len(sample_protein_ids)} sampled Protein nodes")

for ns, ids in protein_by_namespace.items():
    sample_ids = ids[:SAMPLE_PER_NAMESPACE]
    n_resolved = sum(n in protein_sample_sequences for n in sample_ids)
    print(f"  {ns:<12} {n_resolved}/{len(sample_ids)} resolved")


### 4.2 Full protein resolve and embed

The full run resolves and embeds all Protein nodes from the command line. The embedding step benefits from a GPU.

```bash
conda activate plantmetbench-data

python scripts/embed_proteins.py \
    --data-dir data/processed --cache-dir data/interim/full_embeddings \
    --resolve-only --workers 5

python scripts/embed_proteins.py \
    --data-dir data/processed --cache-dir data/interim/full_embeddings \
    --device cuda
# -> data/processed/embeddings_protein.pt
```

Resolution is network-bound, so its duration depends on the number of nodes and on how many namespaces need the slower fallback lookups. The sample in §4.1 gives an early estimate of the per-namespace hit rate. Both stages append to `embedding_log.jsonl`, which §6 reads.

In [ ]:
_prot_cache_path = CACHE_DIR / "protein_sequences.json"
_prot_unres_path = CACHE_DIR / "protein_unresolved.txt"
_prot_total = len(protein_ids)

import json as _json
if _prot_cache_path.exists():
    _prot_cached = _json.loads(_prot_cache_path.read_text())
    _n_res = len(_prot_cached)
    print(f"Protein full-run resolve: {_n_res}/{_prot_total} resolved ({100*_n_res/_prot_total:.1f}%)")
    if _prot_unres_path.exists():
        _unres_ids = [l for l in _prot_unres_path.read_text().splitlines() if l.strip()]
        print(f"  unresolved: {len(_unres_ids)} -> {_prot_unres_path}")
else:
    print("protein_sequences.json not found -- full resolve has not been run yet. See §4.2 above for the command.")


### 4.3 Protein resolution by namespace

Breaking the full-run result down by namespace shows where resolution fails and whether the failures are specific to *A. thaliana*.

In [ ]:
_prot_resolved = _json.loads(_prot_cache_path.read_text()) if _prot_cache_path.exists() else {}

print(f"{'namespace':<12} {'resolved':>9} {'total':>7}  {'%':>6}")
for ns, ids in protein_by_namespace.items():
    n_ok = sum(n in _prot_resolved for n in ids)
    print(f"{ns:<12} {n_ok:>9} {len(ids):>7}  {100*n_ok/len(ids):>5.1f}%")

# tair.name/plantcyc resolution goes through extract_locus() -> a gene-name
# search, restricted to the organism in organism_map. Split that namespace's
# result by organism to test the "only A. thaliana resolves" hypothesis directly.
plantcyc_ids = protein_by_namespace.get("plantcyc", [])
ath_ids   = [n for n in plantcyc_ids if organism_map.get(n) == "3702"]
other_ids = [n for n in plantcyc_ids if organism_map.get(n) not in ("3702", None)]
print(f"\nWithin 'plantcyc' ({len(plantcyc_ids)} nodes):")
print(f"  tagged A. thaliana (3702): {sum(n in _prot_resolved for n in ath_ids)}/{len(ath_ids)} resolved")
print(f"  tagged another species:   {sum(n in _prot_resolved for n in other_ids)}/{len(other_ids)} resolved")

print("\nA few unresolved 'plantcyc' identifiers, with what extract_locus() does to them:")
for n in [n for n in plantcyc_ids if n not in _prot_resolved][:6]:
    print(f"  {common.local_id(n):<28} -> extract_locus: {common.extract_locus(n)!r}")


The failures are not confined to non-Arabidopsis species. `extract_locus()` only succeeds when the raw PlantCyc identifier contains a recognisable `AT#G#####` or `Glyma.##G######` substring (`AT3G22200-LER-MONOMER` gives `AT3G22200`). It fails for:

- *A. thaliana* entries that use a gene symbol instead of a locus tag (`ATGSTF1-MONOMER`)
- generic numeric PlantCyc accessions (`753205-MONOMER`), in any species
- accessions that follow another species' genome-database convention (`CHLREDRAFT_113296-MONOMER` for *C. reinhardtii*, `CA08G10750-MONOMER` for pepper)

When no locus is found, the identifier is passed on unchanged and searched in UniProt as a gene name, which returns nothing. The low `plantcyc` resolution therefore comes from the locus-format allowlist covering only two species, not from a species bias in the pipeline.

A graph-native shortcut does not exist either: `properties_extra.ttl` has an `AlternativeId_Uniprot` cross-reference, but it is not populated for the unresolved `plantcyc` Protein nodes. The `uniprot` namespace resolves directly by accession, does not use `extract_locus()`, and is unaffected.

## 5. GeneProduct resolution diagnostic (sample)

The locus-extraction limitation matters most here. `tair.name` and `plantcyc` nodes from species other than *A. thaliana* and soybean reach NCBI Gene under their raw identifier, which may or may not be a valid gene name for that species. The sample shows the hit rate per namespace. For a namespace with a low rate, check a few `gene_unresolved.txt` entries to tell a locus-format mismatch (fixable by extending `LOCUS_RE`) from a gene that is absent from NCBI.

In [ ]:
SAMPLE_PER_NAMESPACE = 8

gene_ids = common.load_nodes(FULL_DIR, "GeneProduct")
gene_by_namespace = {}
for node_id in gene_ids:
    gene_by_namespace.setdefault(common.namespace(node_id), []).append(node_id)

sample_gene_ids = []
for ns, ids in gene_by_namespace.items():
    sample_gene_ids.extend(ids[:SAMPLE_PER_NAMESPACE])
    print(f"  sampling {min(SAMPLE_PER_NAMESPACE, len(ids))}/{len(ids)} '{ns}' GeneProduct ids")

gene_sample_sequences = embed_genes.resolve_gene_sequences(sample_gene_ids, organism_map, CACHE_DIR)
print(f"\nresolved sequences for {len(gene_sample_sequences)}/{len(sample_gene_ids)} sampled GeneProduct nodes")

for ns, ids in gene_by_namespace.items():
    sample_ids = ids[:SAMPLE_PER_NAMESPACE]
    n_resolved = sum(n in gene_sample_sequences for n in sample_ids)
    print(f"  {ns:<12} {n_resolved}/{len(sample_ids)} resolved")


### 5.2 Full gene resolve and embed

```bash
python scripts/embed_genes.py \
    --data-dir data/processed --cache-dir data/interim/full_embeddings \
    --resolve-only
# NCBI limits requests to about 3 per second without an API key. Setting
# NCBI_API_KEY in the environment triples the limit and allows --workers 3.

python scripts/embed_genes.py \
    --data-dir data/processed --cache-dir data/interim/full_embeddings \
    --device cuda
# -> data/processed/embeddings_gene.pt
```

In [ ]:
_gene_cache_path = CACHE_DIR / "gene_sequences.json"
_gene_unres_path = CACHE_DIR / "gene_unresolved.txt"
_gene_total = len(gene_ids)

if _gene_cache_path.exists():
    _gene_cached = _json.loads(_gene_cache_path.read_text())
    _n_res = len(_gene_cached)
    print(f"GeneProduct full-run resolve: {_n_res}/{_gene_total} resolved ({100*_n_res/_gene_total:.1f}%)")
    if _gene_unres_path.exists():
        _unres_ids = [l for l in _gene_unres_path.read_text().splitlines() if l.strip()]
        print(f"  unresolved: {len(_unres_ids)} -> {_gene_unres_path}")
else:
    print("gene_sequences.json not found -- full resolve has not been run yet. See §5.2 above for the command.")


## 6. Embedding coverage

This section reads `embedding_log.jsonl`, to which every resolve and embed stage appends, so the numbers reflect the runs regardless of where they were launched.

In [ ]:
import json as _json

if not LOG_PATH.exists():
    print(f"No log yet at {LOG_PATH} -- run the resolve/embed commands above first.")
else:
    records = [_json.loads(l) for l in LOG_PATH.read_text().splitlines() if l.strip()]
    # keep only the latest record per (node_type, stage)
    latest = {}
    for r in records:
        latest[(r["node_type"], r["stage"])] = r

    print(f"{'Node type':<14} {'Stage':<10} {'Total':>7} {'Count':>7} {'%':>6}  Timestamp (UTC)")
    print("-" * 70)
    for node_type in ["Metabolite", "Protein", "GeneProduct"]:
        for stage, key in [("resolve", "resolved"), ("embed", "embedded")]:
            r = latest.get((node_type, stage))
            if r is None:
                print(f"{node_type:<14} {stage:<10} {'--':>7} {'--':>7} {'--':>6}  (not run yet)")
                continue
            total, count = r["total"], r[key]
            pct = 100 * count / total if total else 0
            print(f"{node_type:<14} {stage:<10} {total:>7} {count:>7} {pct:>5.1f}%  {r['timestamp_utc']}")


In [ ]:
# Coverage bar chart -- only plots types/stages that have completed (embed) entries.
import matplotlib.pyplot as plt
import numpy as np

if LOG_PATH.exists():
    records = [_json.loads(l) for l in LOG_PATH.read_text().splitlines() if l.strip()]
    embed_records = {r["node_type"]: r for r in records if r["stage"] == "embed"}
    if embed_records:
        types = list(embed_records.keys())
        totals = [embed_records[t]["total"] for t in types]
        embedded = [embed_records[t]["embedded"] for t in types]
        unresolved = [t - e for t, e in zip(totals, embedded)]

        fig, ax = plt.subplots(figsize=(8, 0.8 * len(types) + 1.5))
        y = np.arange(len(types))
        ax.barh(y, embedded, color="#4c72b0", label="will be embedded")
        ax.barh(y, unresolved, left=embedded, color="#dd8452", label="unresolved (no embedding)")
        ax.set_yticks(y)
        ax.set_yticklabels([f"{t}\n({tot})" for t, tot in zip(types, totals)])
        ax.set_xlabel("# nodes")
        ax.set_title("Full graph -- embedding coverage")
        ax.legend(loc="lower right")
        for i, (e, t) in enumerate(zip(embedded, totals)):
            ax.text(t + 10, i, f"{e}/{t} ({100*e/t:.0f}%)", va="center", fontsize=10)
        ax.set_xlim(0, max(totals) * 1.25)
        plt.tight_layout()
        plt.savefig(CACHE_DIR / "coverage_overall.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved -> {CACHE_DIR / 'coverage_overall.png'}")
    else:
        print("No 'embed' stage entries logged yet for any node type -- nothing to plot.")
else:
    print(f"No log yet at {LOG_PATH}.")


## 7.5 Attaching embeddings to HeteroData

`id2idx` is not saved to disk. It is rebuilt from `nodes.tsv` with the same `groupby` order as the graph build step. Unresolved nodes receive a zero vector; they still receive neighbour messages during GNN training, so they are not invisible to the model.

In [ ]:
import torch

id2idx = {ntype: {nid: i for i, nid in enumerate(sub["node_id"].tolist())}
          for ntype, sub in nodes.groupby("node_type")}

data = torch.load(FULL_DIR / "heterodata.pt", weights_only=False)

for emb_file, ntype in [
    ("embeddings_metabolite.pt", "Metabolite"),
    ("embeddings_protein.pt",    "Protein"),
    ("embeddings_gene.pt",       "GeneProduct"),
]:
    emb_path = FULL_DIR / emb_file
    if not emb_path.exists():
        print(f"{ntype}: {emb_path} not found yet -- run its §3-5 embed step first")
        continue
    emb = torch.load(emb_path, weights_only=False)
    dim = next(iter(emb.values())).shape[0]
    # Unresolved nodes -> zero vector (still receive neighbour messages in GNN)
    x = torch.zeros(data[ntype].num_nodes, dim, dtype=torch.float32)
    for node_id, vec in emb.items():
        idx = id2idx[ntype].get(node_id)
        if idx is not None:
            x[idx] = vec.float()   # MAP4 is int64 -- cast to float for GNN layers
    data[ntype].x = x
    print(f"{ntype}: x={tuple(x.shape)}, "
          f"embedded {(x.abs().sum(1) > 0).sum()}/{data[ntype].num_nodes}")

# Persist -- this overwrites the canonical heterodata.pt in place. Safe to
# regenerate from scratch at any time (python scripts/build_pyg.py), and only
# *adds* an `x` attribute per node type -- node identity/order/edges are
# untouched, so every consumer that indexes by node position (splits.pt,
# splits_taxa.pt, splits_pathway.pt, nodes.tsv) stays valid against it.
torch.save(data, FULL_DIR / "heterodata.pt")
print(f"\nSaved -> {FULL_DIR / 'heterodata.pt'} (now includes node features)")


### 7.6 Compatibility check with the splits

`splits_taxa.pt` and `splits_pathway.pt` store edge indices (positions in `Protein` and `Interaction` as ordered in `nodes.tsv` and `heterodata.pt`), not `HeteroData` snapshots, so attaching `x` does not affect them. `splits.pt` (the random split) stores full `RandomLinkSplit` snapshots taken before the embeddings were attached; re-run the random-split cells of `02_build_splits.ipynb` if its `train_data`, `val_data` and `test_data` should carry `x`. The check below confirms that the index-based files are in range.

In [ ]:
for split_file in ["splits_taxa.pt", "splits_pathway.pt", "splits.pt"]:
    split_path = FULL_DIR / split_file
    if not split_path.exists():
        print(f"{split_file}: not found -- run notebooks/02_build_splits.ipynb first")
        continue
    split = torch.load(split_path, weights_only=False)

    if "train_edge_index" in split:  # splits_taxa.pt / splits_pathway.pt: index-only format
        n_prot = data["Protein"].num_nodes
        n_int  = data["Interaction"].num_nodes
        # A split can be empty, and .max() on an empty tensor raises, so
        # empty splits are skipped.
        prot_idx_maxes = [int(split[f"{s}_edge_index"][0].max()) for s in ("train", "val", "test")
                           if split[f"{s}_edge_index"].numel() > 0]
        int_idx_maxes  = [int(split[f"{s}_edge_index"][1].max()) for s in ("train", "val", "test")
                           if split[f"{s}_edge_index"].numel() > 0]
        if not prot_idx_maxes:
            print(f"{split_file}: index-only -- all splits empty, nothing to check")
            continue
        max_prot_idx = max(prot_idx_maxes)
        max_int_idx  = max(int_idx_maxes)
        ok = max_prot_idx < n_prot and max_int_idx < n_int
        print(f"{split_file}: index-only -- max Protein idx {max_prot_idx} < {n_prot}, "
              f"max Interaction idx {max_int_idx} < {n_int}  "
              f"[{'OK, in range' if ok else 'OUT OF RANGE -- rebuild needed'}]")
    else:  # splits.pt: full RandomLinkSplit HeteroData snapshots
        has_x = "x" in split["train"]["Protein"]
        print(f"{split_file}: full HeteroData snapshots -- carries Protein.x: {has_x}  "
              f"[{'has embeddings' if has_x else 'predates embeddings -- re-run 02_build_splits.ipynb to refresh'}]")

## 7.7 Embedding coverage by organism

Protein embedding coverage depends mostly on species (§4.3). Only the `uniprot` and `tair.name` namespaces (the latter *A. thaliana* only) resolve well, while `plantcyc`, which most other species depend on, resolves poorly. A split that picks held-out organisms by target-edge count alone can therefore place its val and test organisms in the near-zero-coverage tail. The histogram below shows the per-organism distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

prot_org = pd.DataFrame({
    "protein_idx": data[("Protein", "organism", "Organism")].edge_index[0].numpy(),
    "organism_idx": data[("Protein", "organism", "Organism")].edge_index[1].numpy(),
}).drop_duplicates()

has_embed = (data["Protein"].x.abs().sum(1) > 0).numpy()
prot_org["embedded"] = prot_org["protein_idx"].map(lambda i: bool(has_embed[i]))

# Per-organism coverage distribution
per_org = prot_org.groupby("organism_idx").agg(n_protein=("protein_idx", "nunique"), n_embed=("embedded", "sum"))
per_org["coverage"] = per_org["n_embed"] / per_org["n_protein"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(per_org["coverage"], bins=20, edgecolor="black", color="steelblue")
ax.set_xlabel("Per-organism Protein embedding coverage")
ax.set_ylabel("Number of organisms")
ax.set_title("Protein embedding coverage per organism")
plt.tight_layout()
fig.savefig(FIG_DIR / "taxa_coverage_vs_embedding.png")
plt.show()
print(f"Saved -> {FIG_DIR / 'taxa_coverage_vs_embedding.png'}")

Coverage is bimodal: most organisms lie near 0% or near 100%. A model evaluated on organisms with almost no Protein embeddings is scored mostly on graph topology (message-passing structure, shared pathway membership) rather than on how well it uses the embeddings, which weakens the claim that it generalises to new species. §7.8 therefore makes embedding coverage an eligibility criterion for held-out organisms, in the same way that *A. thaliana* is always kept in train.

## 7.8 Building the taxa split, embedding-coverage-aware

The split is greedy and balanced. *A. thaliana* is always in train; the remaining organisms are sorted by target-edge count and filled into val until at least 10% of the edges are claimed, then into test for another 10%. An organism is eligible for val or test only if **at least `MIN_ORG_EMBED_COVERAGE` of its Proteins have a real (non-zero) embedding**. Otherwise holding it out would test whether the model can do anything with almost no signal, not species generalisation.

`MIN_ORG_EMBED_COVERAGE = 0.10` is the smallest threshold that fills both size targets (11 val and 69 test organisms, average coverage 0.15 and 0.20). Lower thresholds under-fill both splits, and thresholds of 0.15 and above leave the test split under-filled or empty.

The cell overwrites `data/processed/splits_taxa.pt`; re-run `04_link_prediction.ipynb` afterwards.

In [ ]:
MIN_ORG_EMBED_COVERAGE = 0.10

# Target edge: (Protein, catalyzes, Interaction) is a persisted edge in heterodata.pt.
TARGET_EDGE = ("Protein", "catalyzes", "Interaction")
target_ei = data[TARGET_EDGE].edge_index
prot_int = pd.DataFrame({"protein_idx": target_ei[0].numpy(), "interaction_idx": target_ei[1].numpy()})
n_total = len(prot_int)
print(f"Target edges ({TARGET_EDGE}): {n_total:,}")

# Exclusions: multi-label and fallback-taxon proteins are always placed in train
org_nodes = (
    nodes[nodes["node_type"] == "Organism"]
    .reset_index(drop=True).rename_axis("organism_idx").reset_index()
)
org_nodes["taxon"] = org_nodes["node_id"].str.rsplit("/", n=1).str[-1]
FALLBACK_TAXON = "NCBITaxon_33090"
fallback_org_idx = int(org_nodes.loc[org_nodes["taxon"] == FALLBACK_TAXON, "organism_idx"].iloc[0])

taxa_per_protein = prot_org.groupby("protein_idx")["organism_idx"].agg(set)
multi_label_proteins = set(taxa_per_protein[taxa_per_protein.apply(len) > 1].index)
fallback_proteins = set(taxa_per_protein[taxa_per_protein.apply(lambda s: s == {fallback_org_idx})].index)
excluded_proteins = multi_label_proteins | fallback_proteins
eligible_prot_org = prot_org[~prot_org["protein_idx"].isin(excluded_proteins)]
edge_org = prot_int.merge(eligible_prot_org, on="protein_idx")

org_edge_counts = edge_org.groupby("organism_idx").size().reset_index(name="n_target_edges")
org_stats = org_edge_counts.merge(org_nodes[["organism_idx", "taxon"]], on="organism_idx", how="left")
org_stats = org_stats.merge(per_org[["coverage"]], on="organism_idx", how="left")
org_stats["coverage"] = org_stats["coverage"].fillna(0.0)
org_stats = org_stats.sort_values("n_target_edges", ascending=False)

ALWAYS_TRAIN_TAXA = {"NCBITaxon_3702"}
always_train_idx = set(org_nodes[org_nodes["taxon"].isin(ALWAYS_TRAIN_TAXA)]["organism_idx"].values)

# Embedding-coverage eligibility gate for the greedy walk
assignable = org_stats[
    (~org_stats["organism_idx"].isin(always_train_idx))
    & (org_stats["coverage"] >= MIN_ORG_EMBED_COVERAGE)
].sort_values("n_target_edges", ascending=False)

val_target = 0.10 * n_total
test_target = 0.10 * n_total
val_orgs, test_orgs = [], []
val_count, test_count = 0, 0
for _, row in assignable.iterrows():
    oid, n = row["organism_idx"], row["n_target_edges"]
    if val_count < val_target:
        val_orgs.append(oid); val_count += n
    elif test_count < test_target:
        test_orgs.append(oid); test_count += n

val_set, test_set = set(val_orgs), set(test_orgs)
train_set = set(org_stats["organism_idx"].values) - val_set - test_set

edge_orgs_grouped = (
    edge_org.groupby(["protein_idx", "interaction_idx"])["organism_idx"]
    .apply(frozenset).reset_index(name="orgs")
)

def assign_split(orgs):
    if orgs & val_set: return "val"
    elif orgs & test_set: return "test"
    else: return "train"

edge_orgs_grouped["split"] = edge_orgs_grouped["orgs"].apply(assign_split)

unassigned_mask = ~prot_int.set_index(["protein_idx", "interaction_idx"]).index.isin(
    edge_orgs_grouped.set_index(["protein_idx", "interaction_idx"]).index
)
unassigned_df = prot_int[unassigned_mask].copy()
unassigned_df["orgs"] = [frozenset()] * len(unassigned_df)
unassigned_df["split"] = "train"
all_edges = pd.concat([edge_orgs_grouped, unassigned_df], ignore_index=True)

sc = all_edges["split"].value_counts()
print(f"\nSplit (MIN_ORG_EMBED_COVERAGE={MIN_ORG_EMBED_COVERAGE}):")
for s in ["train", "val", "test"]:
    print(f"  {s:5s}: {sc.get(s,0):6,}  ({100*sc.get(s,0)/n_total:.1f}%)")

def org_cov_summary(name, idx_set):
    rows = org_stats[org_stats["organism_idx"].isin(idx_set)]
    avg_cov = rows["coverage"].mean() if len(rows) else float("nan")
    print(f"{name}: {len(idx_set)} organisms, avg Protein embed coverage = {avg_cov:.2f}")

org_cov_summary("Val ", val_set)
org_cov_summary("Test", test_set)

# Sanity check: protein sets should still be disjoint across splits (single-label organisms)
def split_protein_set(name):
    return set(all_edges[all_edges["split"] == name]["protein_idx"].values)
train_prots, val_prots, test_prots = split_protein_set("train"), split_protein_set("val"), split_protein_set("test")
def set_tanimoto(a, b):
    i, u = len(a & b), len(a | b)
    return i / u if u else 0.0
tan_tv, tan_tt, tan_vt = set_tanimoto(train_prots, val_prots), set_tanimoto(train_prots, test_prots), set_tanimoto(val_prots, test_prots)
print(f"\nCross-split Tanimoto (protein sets, expect ~0): train-val={tan_tv:.4f} train-test={tan_tt:.4f} val-test={tan_vt:.4f}")

def make_edge_index(name):
    sub = all_edges[all_edges["split"] == name]
    return torch.stack([
        torch.tensor(sub["protein_idx"].values, dtype=torch.long),
        torch.tensor(sub["interaction_idx"].values, dtype=torch.long),
    ], dim=0)

train_ei, val_ei, test_ei = make_edge_index("train"), make_edge_index("val"), make_edge_index("test")

torch.save(
    {
        "train_edge_index": train_ei, "val_edge_index": val_ei, "test_edge_index": test_ei,
        "target_edge": TARGET_EDGE,
        "train_organisms": sorted(train_set), "val_organisms": sorted(val_set), "test_organisms": sorted(test_set),
        "split_params": {
            "strategy": "taxa_greedy_balanced_embedding_aware",
            "always_train_taxa": sorted(ALWAYS_TRAIN_TAXA),
            "min_org_embed_coverage": MIN_ORG_EMBED_COVERAGE,
            "val_target_frac": 0.10, "test_target_frac": 0.10,
        },
        "tanimoto": {"train_val": float(tan_tv), "train_test": float(tan_tt), "val_test": float(tan_vt)},
    },
    FULL_DIR / "splits_taxa.pt",
)
print(f"\nSaved -> {FULL_DIR / 'splits_taxa.pt'}")
print(f"  train: {train_ei.shape[1]:,}  val: {val_ei.shape[1]:,}  test: {test_ei.shape[1]:,}")

### 7.9 Final check: embedding coverage and organism overlap per split

Two checks on the final split:

1. Embedding coverage per split for every node type the task touches: Protein, GeneProduct (via `encodes`) and Metabolite (via the Conversion's `target` and `participants`). Apparent performance on a split depends on how much real signal the model has for all of them.
2. Organism overlap. The train, val and test organism sets are disjoint by construction; the cell prints the set sizes and their intersections.

In [ ]:
has_embed_gene = (data["GeneProduct"].x.abs().sum(1) > 0).numpy()
has_embed_met  = (data["Metabolite"].x.abs().sum(1) > 0).numpy()

encodes_ei = data[("GeneProduct", "encodes", "Protein")].edge_index
gene_of_protein = {}
for g, p in zip(encodes_ei[0].tolist(), encodes_ei[1].tolist()):
    gene_of_protein.setdefault(int(p), []).append(int(g))

met_of_interaction = {}
for rel in ["target", "source", "participants"]:
    et = ("Interaction", rel, "Metabolite")
    if et not in data.edge_types:
        continue
    ei = data[et].edge_index
    for i, m in zip(ei[0].tolist(), ei[1].tolist()):
        met_of_interaction.setdefault(int(i), set()).add(int(m))

split_ei = {"train": train_ei, "val": val_ei, "test": test_ei}
split_orgs = {"train": train_set, "val": val_set, "test": test_set}

print(f"{'Split':<6} {'Protein':>16} {'GeneProduct':>16} {'Metabolite':>16}")
for name, ei in split_ei.items():
    prot_idxs = sorted(set(ei[0].tolist()))
    int_idxs  = sorted(set(ei[1].tolist()))

    n_prot = len(prot_idxs)
    n_prot_emb = int(has_embed[prot_idxs].sum()) if n_prot else 0

    gene_idxs = sorted({g for p in prot_idxs for g in gene_of_protein.get(p, [])})
    n_gene = len(gene_idxs)
    n_gene_emb = int(has_embed_gene[gene_idxs].sum()) if n_gene else 0

    met_idxs = sorted({m for i in int_idxs for m in met_of_interaction.get(i, set())})
    n_met = len(met_idxs)
    n_met_emb = int(has_embed_met[met_idxs].sum()) if n_met else 0

    def fmt(n_emb, n):
        return f"{n_emb}/{n} ({100*n_emb/n:.0f}%)" if n else "0/0 (--)"

    print(f"{name:<6} {fmt(n_prot_emb, n_prot):>16} {fmt(n_gene_emb, n_gene):>16} {fmt(n_met_emb, n_met):>16}")

print()
print("Taxa overlap (organism sets, by construction every organism belongs to exactly one split):")
print(f"  train: {len(split_orgs['train'])} organisms")
print(f"  val  : {len(split_orgs['val'])} organisms")
print(f"  test : {len(split_orgs['test'])} organisms")
print(f"  train ∩ val : {len(split_orgs['train'] & split_orgs['val'])} (expect 0)")
print(f"  train ∩ test: {len(split_orgs['train'] & split_orgs['test'])} (expect 0)")
print(f"  val ∩ test  : {len(split_orgs['val'] & split_orgs['test'])} (expect 0)")

fig, ax = plt.subplots(figsize=(7, 4.5))
splits_lbl = ["train", "val", "test"]
width = 0.25
x = np.arange(len(splits_lbl))
for off, (ntype, has_emb_arr, idx_key) in enumerate([
    ("Protein", has_embed, 0), ("GeneProduct", has_embed_gene, "gene"), ("Metabolite", has_embed_met, "met"),
]):
    pcts = []
    for name in splits_lbl:
        ei = split_ei[name]
        if idx_key == 0:
            idxs = sorted(set(ei[0].tolist()))
        elif idx_key == "gene":
            prot_idxs = sorted(set(ei[0].tolist()))
            idxs = sorted({g for p in prot_idxs for g in gene_of_protein.get(p, [])})
        else:
            int_idxs = sorted(set(ei[1].tolist()))
            idxs = sorted({m for i in int_idxs for m in met_of_interaction.get(i, set())})
        pcts.append(100 * has_emb_arr[idxs].sum() / len(idxs) if idxs else 0.0)
    ax.bar(x + (off - 1) * width, pcts, width, label=ntype)
ax.set_xticks(x)
ax.set_xticklabels(splits_lbl)
ax.set_ylabel("% nodes with a real (non-zero) embedding")
ax.set_ylim(0, 100)
ax.set_title("Embedding coverage by split and node type (taxa split)")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "taxa_split_coverage_by_type.png", dpi=150)
plt.show()
print(f"\nSaved -> {FIG_DIR / 'taxa_split_coverage_by_type.png'}")